<a href="https://colab.research.google.com/github/lilianabs/aggressiveness-detection-mex-spa/blob/main/3_Pipeline_for_experimenting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# %pip install emoji

In [4]:
# %pip install spacy && python -m spacy download es_core_news_sm

In [5]:
import pandas as pd

pd.set_option("display.max_colwidth", None)

import emoji
import re
import spacy

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, cross_val_score, StratifiedKFold

In [7]:
from google.colab import drive

drive.mount("/content/drive")

nlp = spacy.load("es_core_news_sm")

Mounted at /content/drive


In [8]:
file_path = "/content/drive/MyDrive/data_colab/MEX-A3T/train_aggressiveness.csv"
df = pd.read_csv(file_path)
df = df.drop("Id", axis=1)
display(df.head())

,Category,Text
0,0,Soy el Clint Eastwood de los Puentes de Madison en todas las putas historias de amor que me han tocado.\n
1,0,"Actualmente ya pasó de moda la pucha joto, ahora sólo quedamos los verdaderos seguidores, los que amamos este estilo de vida.\n"
2,0,"¿Es cierto esto? Y no me refiero a lo que dijo, ni al tweet, sino a👉<URL>¿Están tratando de sacarlo del programa?\n"
3,0,Vuela pega y esquiva... la neta está de la vergaaaa pero es pegajosa!!\n
4,0,Mejor puto disfraz de la noche!!!! 👊👊👊Por tercer año consecutivo morros\n


## Library for text cleaning

In [9]:
import re
import string
from typing import Optional, List, Any

import spacy
from sklearn.base import BaseEstimator, TransformerMixin


class TextCleaner(BaseEstimator, TransformerMixin):
    def __init__(self) -> None:
        pass

    def fit(self, X: List[str], y: Optional[Any] = None) -> None:
        return self

    def transform(self, X: List[str], y: Optional[Any] = None) -> List[str]:
        return X


class RemoveEmojis(TextCleaner):
    def transform(self, X: List[str], y: Optional[Any] = None) -> List[str]:
        return [emoji.replace_emoji(text, replace="") for text in X]


class Lowercase(TextCleaner):
    def transform(self, X: List[str], y: Optional[Any] = None) -> List[str]:
        return [text.lower() for text in X]


class RemoveWord(TextCleaner):
    def __init__(self, word: str):
        self.word = word

    def transform(self, X: List[str], y: Optional[Any] = None) -> List[str]:
        return [text.replace(self.word, "") for text in X]


class KeepAlphaNumericChars(TextCleaner):
    def transform(self, X: List[str], y: Optional[Any] = None) -> List[str]:
        return [re.sub(r"[^a-zA-Z0-9\s]", "", text) for text in X]


class RemovePunctuation(TextCleaner):
    def transform(self, X: List[str], y: Optional[Any] = None) -> List[str]:
        table = str.maketrans("", "", string.punctuation)
        return [text.translate(table) for text in X]


class RemoveMentions(TextCleaner):
    def transform(self, X: List[str], y: Optional[Any] = None) -> List[str]:
        return [re.sub(r"@\w+", "", s) for s in X]

## Split dataset into test and train

In [10]:
X = df["Text"]
y = df["Category"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

## Train pipeline

In [11]:
kf = StratifiedKFold(n_splits=5, random_state=42, shuffle=True)
target_names = ["0", "1"]

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    pipeline = Pipeline(
        [
            ("remove_mentions", RemoveMentions()),
            ("remove_emojis", RemoveEmojis()),
            ("remove_unnecessary_words", RemoveWord("<URL>")),
            ("remove_line_change", RemoveWord("\n")),
            ("lower_case", Lowercase()),
            ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
            ("lr", LogisticRegression(random_state=42)),
        ]
    )

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_val)

    print(f"Fold {fold + 1} classification report")
    print(classification_report(y_val, y_pred, target_names=target_names))
    print("-" * 50)

    # print f1 macro avg instead of classification report

Fold 1 classification report
              precision    recall  f1-score   support

           0       0.82      0.98      0.89      1045
           1       0.90      0.48      0.63       422

    accuracy                           0.84      1467
   macro avg       0.86      0.73      0.76      1467
weighted avg       0.85      0.84      0.82      1467

--------------------------------------------------
Fold 2 classification report
              precision    recall  f1-score   support

           0       0.81      0.96      0.88      1045
           1       0.82      0.46      0.59       422

    accuracy                           0.82      1467
   macro avg       0.82      0.71      0.73      1467
weighted avg       0.82      0.82      0.80      1467

--------------------------------------------------
Fold 3 classification report
              precision    recall  f1-score   support

           0       0.81      0.96      0.88      1044
           1       0.83      0.45      0.58     